#Projet Python M2

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import pickle

In [20]:
try:
  df_merged = pd.read_csv("/content/merged_69.csv", sep=";") # chemin spécifier
  #print(df_merged.columns)
except FileNotFoundError:
  print("Le chemin n'est pas bon")

<ipython-input-20-8a89467cbcdc>:2: DtypeWarning: Columns (33,35,38,51,57,69,72,99,115) have mixed types. Specify dtype option on import or set low_memory=False.
  df_merged = pd.read_csv("/content/merged_69.csv", sep=";") # chemin spécifier


In [21]:
# MLFlow ?
df_merged.head()

,Ubat_W/m²_K,Complément_d'adresse_bâtiment,Méthode_application_DPE,N°_département_(BAN),Qualité_isolation_menuiseries,Nombre_appartement,Emission_GES_5_usages_énergie_n°2,Coordonnée_cartographique_X_(BAN),Date_fin_validité_DPE,Protection_solaire_exterieure_(0/1),...,Modèle_DPE,Coût_auxiliaires,Coût_refroidissement_dépensier,Nom__rue_(BAN),Nom__commune_(BAN),Emission_GES_ECS_dépensier,Conso_5_usages_par_m²_é_primaire,Besoin_chauffage,Appartement_non_visité_(0/1),Complément_d'adresse_logement
0,1.42,NaN,dpe appartement individuel,69.0,moyenne,1.0,NaN,842365.11,2034-08-28,False,...,DPE 3CL 2021 méthode logement,0.0,0.0,Rue Imbert Colomès,Lyon,117.1,535.0,5261.0,False,1-appartement-1-LogZone
1,2.36,653199,dpe appartement individuel,69.0,moyenne,1.0,NaN,842432.78,2034-07-23,False,...,DPE 3CL 2021 méthode logement,68.6,0.0,Rue Longue,Lyon,107.4,583.0,9089.0,NaN,2ième Etage Porte Gauche
2,0.59,2024-08-15639,dpe appartement individuel,69.0,moyenne,1.0,16.7,842540.42,2034-08-11,True,...,DPE 3CL 2021 méthode logement,62.5,0.0,Rue Imbert Colomès,Lyon,387.9,386.0,8784.5,NaN,Etage 6
3,0.42,23/DFA/0199,dpe appartement individuel,69.0,moyenne,1.0,34.5,842600.66,2034-07-28,True,...,DPE 3CL 2021 méthode logement,135.1,0.0,Rue Romarin,Lyon,465.0,156.0,5039.0,NaN,NaN
4,1.29,240911838,dpe appartement individuel,69.0,bonne,1.0,NaN,842544.94,2034-09-09,False,...,DPE 3CL 2021 méthode logement,19.6,0.0,Rue Saint Polycarpe,Lyon,75.8,152.0,779.6,NaN,Etage 6


In [22]:
df_merged.shape


(287165, 131)

In [23]:
# Afficher la liste complète des noms de colonnes pour identifier les cibles "étiquettes DPE" et "consommation énergétique"
columns = df_merged.columns.tolist()
columns

['Ubat_W/m²_K',
 "Complément_d'adresse_bâtiment",
 'Méthode_application_DPE',
 'N°_département_(BAN)',
 'Qualité_isolation_menuiseries',
 'Nombre_appartement',
 'Emission_GES_5_usages_énergie_n°2',
 'Coordonnée_cartographique_X_(BAN)',
 'Date_fin_validité_DPE',
 'Protection_solaire_exterieure_(0/1)',
 'Coût_total_5_usages',
 'Conso_ECS_dépensier_é_primaire',
 'Catégorie_ENR',
 'Présence_brasseur_air_(0/1)',
 'Date_établissement_DPE',
 'Score_BAN',
 'Type_énergie_n°2',
 'Conso_auxiliaires_é_primaire',
 'N°_étage_appartement',
 'N°_région_(BAN)',
 'Deperditions_planchers_hauts',
 'Adresse_brute',
 'Emission_GES_refroidissement_dépensier',
 'Version_DPE',
 'Deperditions_planchers_bas',
 'Surface_tertiaire_immeuble',
 'Coût_chauffage',
 '_i',
 'Qualité_isolation_plancher_bas',
 'Déperditions_renouvellement_air',
 'Surface_habitable_logement',
 'Type_énergie_n°1',
 'Conso_chauffage_é_finale',
 'Besoin_refroidissement',
 'Emission_GES_éclairage',
 'Code_INSEE_(BAN)',
 'Conso_refroidissement_

In [24]:
df1 = df_merged

In [25]:
df1.shape

(287165, 131)

In [26]:
pd.set_option('display.max_rows', None)

In [27]:
#Fonction pour identifier le % de valeurs manquantes dans chaque colonne
#Renvoie seulement les colonnes contenant des valeurs manquantes + les tri
def percent_missing(df):
  percent_nan = 100 * df.isnull().sum() / len(df)
  percent_nan = percent_nan[percent_nan > 0].sort_values()

  return percent_nan

percent_nan = percent_missing(df1)
percent_nan

,0
Ubat_W/m²_K,0.000348
Conso_auxiliaires_é_finale,0.000348
Conso_chauffage_é_primaire,0.002089
Emission_GES_chauffage,0.002089
Conso_chauffage_dépensier_é_finale,0.002089
Conso_chauffage_dépensier_é_primaire,0.002089
Conso_chauffage_é_finale,0.002089
Emission_GES_chauffage_dépensier,0.002089
Coût_éclairage,0.002438
Conso_5_usages_é_finale,0.002786


In [28]:
#suppression des colonnes qui ont + de > 30% valeurs manquantes
df1 = df1.drop(percent_nan[percent_nan > 30].index, axis=1)

In [29]:
percent_nan = percent_missing(df1)
percent_nan

,0
Ubat_W/m²_K,0.000348
Conso_auxiliaires_é_finale,0.000348
Conso_chauffage_é_primaire,0.002089
Conso_chauffage_é_finale,0.002089
Emission_GES_chauffage,0.002089
Conso_chauffage_dépensier_é_primaire,0.002089
Conso_chauffage_dépensier_é_finale,0.002089
Emission_GES_chauffage_dépensier,0.002089
Coût_éclairage,0.002438
Conso_5_usages_é_finale,0.002786


In [32]:
#supprimer les lignes, dans les colonnes qui ont moins de 9% de valeurs manquantes
df1 = df1.dropna(subset=percent_nan[percent_nan < 9].index, axis=0)

In [33]:
percent_nan = percent_missing(df1)
percent_nan

,0
Type_installation_chauffage,9.713755
Type_installation_ECS_(général),9.713755
Nombre_appartement,10.340823
Complément_d'adresse_logement,17.877785


In [34]:
#Imputation des Données Manquantes sur variable qualitative

# Sélectionner toutes les colonnes non numériques (qualitatives)
categorical_cols = df1.select_dtypes(exclude=[np.number]).columns

# Appliquer l'imputation par la valeur la plus fréquente (mode) pour chaque colonne catégorielle
for col in categorical_cols:
    df1[col] = df1[col].fillna(df1[col].mode()[0])

# Vérification des données manquantes
#df1[categorical_cols].isnull().sum()

In [35]:
col = df1.columns.tolist()
col
# Sélectionner toutes les colonnes sauf celles contenant "Coût" ou "Conso" ou "GES" qui sont liées a la variable target pour éviter Data Leakage
les_variables_explicatives = [col for col in df1.columns if not ("Coût" in col or "Conso" in col or "GES" in col or "coût" in col)]
#les_variables_explicatives

In [36]:
# Filtrer les colonnes quantitatives parmi les variables explicatives
quantitative_variables = [col for col in les_variables_explicatives if np.issubdtype(df1[col].dtype, np.number)]
# Afficher les variables quantitatives sélectionnées
#print("Variables quantitatives parmi les variables explicatives :", quantitative_variables)
quantitative_variables

['Ubat_W/m²_K',
 'N°_département_(BAN)',
 'Nombre_appartement',
 'Coordonnée_cartographique_X_(BAN)',
 'Score_BAN',
 'N°_étage_appartement',
 'N°_région_(BAN)',
 'Deperditions_planchers_hauts',
 'Version_DPE',
 'Deperditions_planchers_bas',
 '_i',
 'Déperditions_renouvellement_air',
 'Surface_habitable_logement',
 '_score',
 'Nombre_niveau_logement',
 'Déperditions_ponts_thermiques',
 'Coordonnée_cartographique_Y_(BAN)',
 'Deperditions_enveloppe',
 'Hauteur_sous-plafond',
 'Déperditions_portes',
 'Code_postal_(BAN)',
 '_rand',
 'Déperditions_murs',
 'Deperditions_baies_vitrées',
 'Code_postal_(brut)',
 'Besoin_chauffage']

In [37]:
# Supprimer les colonnes constantes (qui ne donneront donc aucune corrélation)
quantitative_variables = [col for col in quantitative_variables if df1[col].nunique() > 1]
# Calculer la corrélation entre chaque variable quantitative et 'Conso_5_usages_é_finale'
correlations = df1[quantitative_variables].corrwith(df1['Conso_5_usages_é_finale'])
# Trier les corrélations dans l'ordre décroissant
correlations_sorted = correlations.sort_values(ascending=False)
# Afficher les corrélations triées
#print(correlations_sorted)
# Filtrer pour garder uniquement les corrélations supérieures à 0.1
correlations_filtered = correlations_sorted[correlations_sorted > 0.1]
# Afficher les corrélations filtrées
print(correlations_filtered)

Surface_habitable_logement    0.664904
Ubat_W/m²_K                   0.324088
_score                        0.198399
Code_postal_(BAN)             0.161749
Nombre_niveau_logement        0.130350
dtype: float64


In [38]:
df1['Conso_5_usages_é_finale'].describe()
#Il faut traiter les outliers de df1['Conso_5_usages_é_finale'] pour éviter que le modèle apprenne mal

,Conso_5_usages_é_finale
count,189453.000000
mean,10399.339109
std,9198.048501
min,556.500000
25%,5015.400000
50%,8032.000000
75%,13158.900000
max,429196.900000


In [39]:
EI = 13158.900000 - 5015.400000
limitSup = 13158.900000 + (EI * 1.5)
print(limitSup) #le seuil qui considère valeurs aberrantes niv sup

# Compter le nombre de valeurs supérieures à 20000 (outliers)
count_above_20000 = (df1['Conso_5_usages_é_finale'] > 25000).sum()

# Afficher le résultat
print("Nombre de valeurs supérieures à 25000 :", count_above_20000)

# Calculer le pourcentage
percentage_above_20000 = (count_above_20000 / len(df1['Conso_5_usages_é_finale'])) * 100

# Afficher le résultat
print("Pourcentage de valeurs supérieures à 25000 :", percentage_above_20000, "%")

25374.15
Nombre de valeurs supérieures à 25000 : 9172
Pourcentage de valeurs supérieures à 25000 : 4.841306287047447 %


In [40]:
df2 = df1
df2 = df2[df2['Conso_5_usages_é_finale'] <= 25000] #suppression des outliers qui représentent 5% de la colonne

In [41]:
import scipy.stats as stats
#faire test stat pour évaluer corrélation entre variable cible et les var qualitatitves

quantitative_var = 'Conso_5_usages_é_finale'
# Filtrer les variables qualitatives
qualitative_vars = df2.select_dtypes(include=['object', 'category']).columns
# Dictionnaire pour stocker les résultats
anova_results = {}

# Parcourir les variables qualitatives
for qual_var in qualitative_vars:
    # Groupement des données en une seule opération
    grouped_data = df2.groupby(qual_var)[quantitative_var].apply(list)

    # Filtrer les groupes avec au moins deux valeurs
    filtered_groups = [group for group in grouped_data if len(group) > 1]

    # Effectuer le test ANOVA si le nombre de groupes est supérieur à 1
    if len(filtered_groups) > 1:
        f_stat, p_value = stats.f_oneway(*filtered_groups)
        anova_results[qual_var] = {'F_statistic': f_stat, 'p_value': p_value}


In [42]:
# Trier les résultats par ordre décroissant de la statistique F
anova_results_sorted = dict(sorted(anova_results.items(), key=lambda item: item[1]['F_statistic'], reverse=True))
# Afficher les résultats triés
for var, result in anova_results_sorted.items():
    print(f"Variable qualitative : {var}")
    print(f"  Statistique F : {result['F_statistic']:.4f}, Valeur p : {result['p_value']:.4e}\n")

Variable qualitative : Etiquette_GES
  Statistique F : 20243.1961, Valeur p : 0.0000e+00

Variable qualitative : Qualité_isolation_murs
  Statistique F : 12228.2173, Valeur p : 0.0000e+00

Variable qualitative : Type_bâtiment
  Statistique F : 11322.7571, Valeur p : 0.0000e+00

Variable qualitative : Qualité_isolation_enveloppe
  Statistique F : 7991.5052, Valeur p : 0.0000e+00

Variable qualitative : Méthode_application_DPE
  Statistique F : 5687.0294, Valeur p : 0.0000e+00

Variable qualitative : Etiquette_DPE
  Statistique F : 5251.1286, Valeur p : 0.0000e+00

Variable qualitative : Type_installation_chauffage
  Statistique F : 5070.7517, Valeur p : 0.0000e+00

Variable qualitative : Type_énergie_principale_chauffage
  Statistique F : 4389.2769, Valeur p : 0.0000e+00

Variable qualitative : Qualité_isolation_plancher_bas
  Statistique F : 4186.2469, Valeur p : 0.0000e+00

Variable qualitative : Type_énergie_n°1
  Statistique F : 3876.7200, Valeur p : 0.0000e+00

Variable qualitative

In [43]:
#choix des variable qualitative pour tester modele
label_encoder = LabelEncoder()
df2.loc[:, 'Etiquette_DPE'] = label_encoder.fit_transform(df2['Etiquette_DPE'])
df2.loc[:, 'Type_énergie_principale_chauffage'] = label_encoder.fit_transform(df2['Type_énergie_principale_chauffage'])

In [45]:
#df2.loc[:, 'Qualité_isolation_murs'] = label_encoder.fit_transform(df2['Qualité_isolation_murs'])

In [ ]:
#df2.loc[:, 'Type_bâtiment'] = label_encoder.fit_transform(df2['Type_bâtiment'])

In [56]:
df2.to_csv('df2.csv', index=False)

### Test Régression

In [ ]:
#Test pour régression
#avec seulement les valeurs quantitatives en valeur explicative


# Reshape X pour qu'il soit à 2 dimensions
X = df2[['Surface_habitable_logement', 'Ubat_W/m²_K','Etiquette_DPE', 'Type_énergie_principale_chauffage']]  # Utilisez des doubles crochets pour obtenir un DataFrame
Y = df2['Conso_5_usages_é_finale']

# Séparation des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=42)

# Modèle de régression linéaire
lr_model = LinearRegression()
lr_model = lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

print("MAE : " + str(mean_absolute_error(y_test, y_pred)))
print("RMSE : " + str(root_mean_squared_error(y_test, y_pred)))
print("R² : " + str(r2_score(y_test, y_pred)))


MAE : 2055.1412612420527
RMSE : 2719.035322766427
R² : 0.7336178684502715


In [ ]:

# Créer et entraîner un modèle de forêt aléatoire
rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# Prédictions sur l'ensemble de test
y_pred = rf_model.predict(X_test)

# Calcul des métriques
print("RMSE : " + str(root_mean_squared_error(y_test, y_pred)))
print("MAE : " + str(mean_absolute_error(y_test, y_pred)))
print("R2 : " + str(r2_score(y_test, y_pred)))

RMSE : 2009.7768992211736
MAE : 1446.9347217430757
R2 : 0.8544637799994339


In [ ]:

for col in X_train.select_dtypes(include='object').columns:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')
# Entraîner le modèle
my_model = XGBRegressor(enable_categorical=True)
my_model.fit(X_train, y_train)

# Prédictions sur les ensembles de test et de train
y_pred_test = my_model.predict(X_test)
y_pred_train = my_model.predict(X_train)

# Calcul des métriques pour l'ensemble de test
rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

# Calcul des métriques pour l'ensemble de train
rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

# Affichage des métriques
print("Métriques sur l'ensemble de Test :")
print("RMSE :", rmse_test)
print("MAE :", mae_test)
print("R2 :", r2_test)

print("\nMétriques sur l'ensemble de Train :")
print("RMSE :", rmse_train)
print("MAE :", mae_train)
print("R2 :", r2_train)


Métriques sur l'ensemble de Test :
RMSE : 1312.905349967081
MAE : 897.4199977475859
R2 : 0.9378927604472356

Métriques sur l'ensemble de Train :
RMSE : 1240.066611133793
MAE : 860.9378924790313
R2 : 0.9445630309897123


In [ ]:

# Exporter le modèle dans un fichier avec pickle
with open('linearReg_model.pkl', 'wb') as file:
    pickle.dump(my_model, file)
